In [ ]:
import os
import re
import unicodedata
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import KFold

# 尝试导入 SnowNLP 库用于情感分析
try:
    from snownlp import SnowNLP
    _HAVE_SNOW = True
except ImportError:
    _HAVE_SNOW = False

# ---------- 加载数据 ----------
fname = "ruc_Class25Q2_train_price.csv"  # 数据文件名
if not os.path.exists(fname):
    raise FileNotFoundError(f"{fname} 不存在，请确认文件路径。")

# 尝试读取 CSV 文件，优先使用 UTF-8 编码，如果失败则使用 GBK 编码
try:
    df = pd.read_csv(fname, dtype=str, low_memory=False)
except Exception:
    df = pd.read_csv(fname, dtype=str, encoding='gbk', low_memory=False)

# ---------- 基础函数 ----------
def normalize_text(x):
    """对文本进行规范化处理，包括去除多余的空白字符和特殊字符。"""
    if pd.isna(x): 
        return ""
    s = str(x)
    s = unicodedata.normalize("NFKC", s)  # 将字符规范化
    s = re.sub(r'[\u200B-\u200F\uFEFF]', '', s)  # 去除不可见字符
    return re.sub(r'\s+', ' ', s).strip()  # 简化空格并去掉首尾空格

def doc_sentiment_snownlp(text):
    """计算文本的情感分数及句子统计信息。"""
    if not _HAVE_SNOW:
        return 0.5, 0, 0, 0  # 若未安装 SnowNLP，返回默认值
    s = SnowNLP(text)
    sentiment_score = s.sentiments  # 获取情感得分
    return sentiment_score, int(sentiment_score > 0.6), int(sentiment_score < 0.4), 1  # 返回情感分数和句子统计

# ---------- 特征提取 ----------
def tfidf_svd_fit_transform(texts, max_features=5000, n_components=20, random_state=42):
    """生成TF-IDF特征并进行SVD降维。"""
    tf = TfidfVectorizer(max_features=max_features, token_pattern=r"(?u)\b\w+\b")
    Xtf = tf.fit_transform(texts)  # 进行TF-IDF转换
    svd = TruncatedSVD(n_components=n_components, random_state=random_state)  # SVD降维
    Xred = svd.fit_transform(Xtf)  # 进行SVD变换
    return Xred, tf, svd  # 返回降维后的特征和模型

def oof_group_aggregate(train_df, group_col, feat_cols, n_splits=5, seed=42):
    """使用KFold进行OOF聚合以生成组级文本特征。"""
    oof_dict = {}
    global_means = {}
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)

    for f in feat_cols:
        # 确保每一列只包含数值
        train_df[f] = train_df[f].apply(
            lambda x: x if isinstance(x, (float, int)) else float('nan')  # 非数值数据转为NaN
        )

        # 计算组级特征的均值
        oof = pd.Series(index=train_df.index, dtype=float)
        global_means[f] = train_df[f].mean()  # 全局均值

        for tr, val in kf.split(train_df):
            # 分组计算均值，处理 NaN
            grp_mean = train_df.iloc[tr].groupby(group_col)[f].mean()
            oof.iloc[val] = train_df.iloc[val][group_col].map(grp_mean).fillna(global_means[f])
        
        oof_dict[f] = oof.fillna(global_means[f])  # 若该组无数据，则使用全局均值

    return oof_dict

# ---------- 主流程 ----------
group_col = None
for cand in ['小区', '小区名', '板块', '板块名']:
    if cand in df.columns:
        group_col = cand
        break

if group_col is None:
    raise KeyError("未找到用于聚合的列（如 '小区' 或 '板块'），请确认列名。")

if 'Price' in df.columns:
    try:
        df['Price'] = df['Price'].astype(str).str.replace(',', '').str.replace('元', '').astype(float)
    except Exception:
        df['Price'] = pd.to_numeric(df['Price'].astype(str).str.replace(r'[^\d\.]', '', regex=True), errors='coerce')

txt_col = '客户反馈'
if txt_col not in df.columns:
    raise KeyError(f"找不到列 {txt_col}，请确认。")

# 生成基础文本特征
docs = df[txt_col].fillna("").astype(str).map(normalize_text).tolist()

# 情感分析与句子统计
sent_res = [doc_sentiment_snownlp(t) for t in docs]
df['txt_sent_score'] = [r[0] for r in sent_res]  # 情感得分
df['txt_n_sent_pos'] = [r[1] for r in sent_res]   # 正面句数
df['txt_n_sent_neg'] = [r[2] for r in sent_res]   # 负面句数
df['txt_n_sents'] = [r[3] for r in sent_res]      # 总句数

# 文本长度特征
df['txt_len_chars'] = df[txt_col].fillna("").astype(str).str.len()  # 字符数
df['txt_len_words'] = df[txt_col].fillna("").astype(str).str.count(r'\w+')  # 词数（简单近似）

# TF-IDF + SVD特征生成
print("正在生成 TF-IDF -> SVD 特征。")
Xred, _, _ = tfidf_svd_fit_transform(docs)  # 生成特征
for i in range(Xred.shape[1]):
    df[f'tfidf_svd_{i}'] = Xred[:, i]  # 添加SVD特征

# ---------- OOF 聚合 ----------
doc_feats = ['txt_sent_score', 'txt_n_sent_pos', 'txt_n_sent_neg', 'txt_n_sents', 'txt_len_chars', 'txt_len_words']

if 'Price' in df.columns and df['Price'].notna().sum() > 0:
    print("数据包含 Price，使用 KFold OOF 生成组级文本特征（CV-safe）...")
    oof_dict = oof_group_aggregate(df, group_col, doc_feats)
    for f, s in oof_dict.items():
        df[f'group_{f}'] = s.values
else:
    print("无 Price，直接用全表 group mean 映射（非 OOF），注意风险。")
    for f in doc_feats:
        mapping = df.groupby(group_col)[f].mean().to_dict()
        df[f'group_{f}'] = df[group_col].map(mapping).fillna(df[f].mean())

# 最终保存结果
output_file = "ruc_text_features_with_sentiment.csv"  # 输出文件名
df.to_csv(output_file, index=False)  # 将结果保存为 CSV
print(f"文本特征及情感分析结果已生成并保存为 {output_file}.")


In [ ]:
import pandas as pd
import numpy as np

# 加载数据
file_path = "ruc_text_features_with_sentiment.csv"  # 新的输入文件名
data = pd.read_csv(file_path)

# 显示数据的前几行和数据基本信息
print(data.head())
print(data.info())

# 1. 处理缺失值
# 检查缺失值
missing_values = data.isnull().sum()
print("缺失值统计：\n", missing_values)

# 对于数值型数据：用均值填补
for column in data.select_dtypes(include=[np.number]).columns:
    data[column].fillna(data[column].mean(), inplace=True)

# 对于文本型数据：用'未知'填补
for column in data.select_dtypes(include=[object]).columns:
    data[column].fillna('未知', inplace=True)

# 对于日期型数据：用最近的有效日期填补
for column in data.select_dtypes(include=[np.datetime64]).columns:
    data[column].fillna(data[column].max(), inplace=True)

# 2. 删除空白和无效值
data.replace("", np.nan, inplace=True)

# 3. 处理乱码
# 假设如有特定列可能存在编码问题，进行清理：
# data['column_name'] = data['column_name'].str.encode('utf-8').str.decode('utf-8')

# 4. 标准化数据格式
# 标准化日期格式
for column in data.select_dtypes(include=[np.datetime64]).columns:
    data[column] = pd.to_datetime(data[column], errors='coerce')

# 5. 检查并处理异常值
# 这里以价格列为例，进行异常值检测
price_column = "Price"  # 假设你的价格列名为 "price"
q1 = data[price_column].quantile(0.25)
q3 = data[price_column].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

data = data[(data[price_column] >= lower_bound) & (data[price_column] <= upper_bound)]

# 6. 删除重复数据
data.drop_duplicates(inplace=True)

# 7. 数据类型转换
# 确保数值、日期和文本数据的类型正确
for column in data.select_dtypes(include=[object]).columns:
    data[column] = data[column].astype(str)

# 8. 清洗完成后，重新检查数据
print("清洗后的数据:")
print(data.info())
print(data.head())

# 保存清洗后的数据到新的CSV文件
cleaned_file_path = "cleaned_data.csv"  # 你可以根据需要修改输出文件名
data.to_csv(cleaned_file_path, index=False)
print(f"清洗后的数据已保存到 {cleaned_file_path}")


In [ ]:
import pandas as pd

# 读取CSV文件
file_path = 'cleaned_data.csv'
data = pd.read_csv(file_path)

# 需要处理的列名
columns_to_encode = [
    "环线",
    "房屋户型",
    "所在楼层",
    "房屋朝向",
    "建筑结构",
    "装修情况",
    "梯户比例",
    "配备电梯",
    "别墅类型",
    "交易权属",
    "房屋用途",
    "房屋年限",
    "抵押信息",
    "房屋优势",
    "环线位置",
    "产权所属",
]

# 标签编码
for column in columns_to_encode:
    if column in data.columns:
        data[column], _ = pd.factorize(data[column])

# 保存到新的CSV文件
data.to_csv('encod_cleaned_data.csv', index=False, encoding='utf_8_sig')

print("数据量化完成并已保存至 'encod_cleaned_data.csv'")

In [ ]:
import pandas as pd

# 读取CSV文件
file_path = 'encod_cleaned_data.csv'
data = pd.read_csv(file_path)

# 处理“套内面积”列
def process_area(area):
    if area == '未知' or pd.isna(area) or area == '':
        return 0  # 将“未知”和空字符串替换为0
    else:
        # 移除“m²”单位并转为浮点数
        try:
            return float(area[:-2])  # 去掉最后两个字符
        except ValueError:
            return 0  # 如果转换失败，返回0

# 应用处理函数
data['套内面积'] = data['套内面积'].apply(process_area)

# 打印结果以确认
print(data['套内面积'].head())

# 保存处理后的数据到新的CSV文件
data.to_csv('processed_cleaned_data.csv', index=False, encoding='utf_8_sig')

print("套内面积处理完成并已保存至 'processed_cleaned_data.csv'")

In [ ]:
import pandas as pd

# 读取CSV文件
file_path = 'processed_cleaned_data.csv'
data = pd.read_csv(file_path)

# 时间处理
def process_date(date_str):
    try:
        return pd.to_datetime(date_str)
    except (ValueError, TypeError):
        return pd.NaT

data['交易时间'] = data['交易时间'].apply(process_date)  # 将‘交易时间’转换为datetime
data['上次交易'] = data['上次交易'].apply(process_date)  # 将‘上次交易’转换为datetime

# 处理上次交易的缺失值
# 假设缺失的上次交易视为当前交易时间的上一个时间
data['上次交易'].fillna(data['交易时间'], inplace=True)

# 创建一个新列，标记上次交易是否缺失
data['上次交易缺失'] = data['上次交易'].isna().astype(int)

# 特征提取
data['交易年'] = data['交易时间'].dt.year
data['交易月'] = data['交易时间'].dt.month
data['交易日'] = data['交易时间'].dt.day
data['交易星期'] = data['交易时间'].dt.weekday

# 计算上次交易距离
data['上次交易距离'] = (data['交易时间'] - data['上次交易']).dt.days

# 打印结果以确认
print(data[['交易时间', '上次交易', '上次交易缺失', '上次交易距离']].head())

# 保存处理后的数据
data.to_csv('processed_with_time_features.csv', index=False, encoding='utf_8_sig')

print("时间特征处理完成并已保存至 'processed_with_time_features.csv'")


In [ ]:
import pandas as pd

# 读取CSV文件
file_path = 'processed_with_time_features.csv'
data = pd.read_csv(file_path)

def process_area(area):
        try:
            return float(area[:-2])  # 去掉最后两个字符
        except ValueError:
            return 0  # 如果转换失败，返回0

# 应用处理函数
data['建筑面积'] = data['建筑面积'].apply(process_area)

# 打印结果以确认
print(data['建筑面积'].head())

# 保存处理后的数据到新的CSV文件
data.to_csv('new.csv', index=False, encoding='utf_8_sig')

print("处理完成并已保存至 'new.csv'")

In [ ]:
import pandas as pd

# 读取CSV文件
file_path = 'new.csv'
data = pd.read_csv(file_path)

def process_area(area):
    if area == '未知':
        return 0  # 将“未知”替换为0
    else:
        # 移除“m²”单位并转为浮点数
        try:
            return float(area[:-1])  # 去掉最后字符
        except ValueError:
            return 0  # 如果转换失败，返回0

# 应用处理函数
data['房屋总数'] = data['房屋总数'].apply(process_area)
data['楼栋总数'] = data['楼栋总数'].apply(process_area)
data['绿 化 率'] = data['绿 化 率'].apply(process_area)


# 保存处理后的数据到新的CSV文件
data.to_csv('n.csv', index=False, encoding='utf_8_sig')

print("处理完成并已保存至 'n.csv'")

In [ ]:
import pandas as pd
import re

# 读取CSV文件
file_path = 'n.csv'  # 假设正确的文件路径
data = pd.read_csv(file_path)

# 处理建筑年代
def process_architecture_years(year_str):
    # 使用正则表达式提取年份
    years = re.findall(r'\d{4}', year_str)
    
    if len(years) == 1:  # 仅有一个年份
        return int(years[0])  # 返回单一年份
    elif len(years) == 2:  # 有两个年份
        return int((int(years[0]) + int(years[1])) / 2)  # 返回中间值
    else:
        return None  # 如果没有年份，返回None

# 应用处理函数
data['建筑年代'] = data['建筑年代'].apply(process_architecture_years)

# 打印结果以确认
print(data[['建筑年代']].head())

# 保存处理后的数据
data.to_csv('a.csv')
print("建筑年代处理完成并已保存至 'a.csv'")


In [ ]:
import pandas as pd
import re

# 读取CSV文件
file_path = 'a.csv'  # 假设正确的文件路径
data = pd.read_csv(file_path)

# 处理物业费
def process_property_fee(fee_str):
    # 检查是否为“未知”
    if fee_str.strip().lower() == "未知":
        return 0  # 将“未知”视为0
    
    # 提取所有的数值 (整数或小数)
    numbers = re.findall(r'\d*\.?\d+', fee_str)  # 提取所有数字
    
    # 转换为浮点数
    numbers = [float(num) for num in numbers]
    
    if len(numbers) == 1:  # 仅有一个数值
        return numbers[0]  # 返回该数值
    elif len(numbers) == 2:  # 有两个数值
        return sum(numbers) / 2  # 返回中间值
    else:
        return 0  # 如果无法匹配，有其他情况，返回0

# 应用处理函数
data['物 业 费'] = data['物 业 费'].apply(process_property_fee)
data['燃气费'] = data['燃气费'].apply(process_property_fee)
data['供热费'] = data['供热费'].apply(process_property_fee)

# 保存处理后的数据
data.to_csv('b.csv', index=False, encoding='utf_8_sig')

print("物业费处理完成并已保存至 'b.csv'")


In [ ]:
import os
import re
import unicodedata
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import KFold

# 尝试导入 SnowNLP 库用于情感分析
try:
    from snownlp import SnowNLP
    _HAVE_SNOW = True
except ImportError:
    _HAVE_SNOW = False

# ---------- 加载数据 ----------
fname = "b.csv"  # 数据文件名
if not os.path.exists(fname):
    raise FileNotFoundError(f"{fname} 不存在，请确认文件路径。")

# 尝试读取 CSV 文件，优先使用 UTF-8 编码，如果失败则使用 GBK 编码
try:
    df = pd.read_csv(fname, dtype=str, low_memory=False)
except Exception:
    df = pd.read_csv(fname, dtype=str, encoding='gbk', low_memory=False)

# ---------- 基础函数 ----------
def normalize_text(x):
    """对文本进行规范化处理，包括去除多余的空白字符和特殊字符。"""
    if pd.isna(x): 
        return ""
    s = str(x)
    s = unicodedata.normalize("NFKC", s)  # 将字符规范化
    s = re.sub(r'[\u200B-\u200F\uFEFF]', '', s)  # 去除不可见字符
    return re.sub(r'\s+', ' ', s).strip()  # 简化空格并去掉首尾空格

def doc_sentiment_snownlp(text):
    """计算文本的情感分数及句子统计信息。"""
    if not _HAVE_SNOW:
        return 0.5, 0, 0, 0  # 若未安装 SnowNLP，返回默认值
    s = SnowNLP(text)
    sentiment_score = s.sentiments  # 获取情感得分
    return sentiment_score, int(sentiment_score > 0.6), int(sentiment_score < 0.4), 1  # 返回情感分数和句子统计

# ---------- 特征提取 ----------
def tfidf_svd_fit_transform(texts, max_features=5000, n_components=20, random_state=42):
    """生成TF-IDF特征并进行SVD降维。"""
    tf = TfidfVectorizer(max_features=max_features, token_pattern=r"(?u)\b\w+\b")
    Xtf = tf.fit_transform(texts)  # 进行TF-IDF转换
    svd = TruncatedSVD(n_components=n_components, random_state=random_state)  # SVD降维
    Xred = svd.fit_transform(Xtf)  # 进行SVD变换
    return Xred, tf, svd  # 返回降维后的特征和模型

def oof_group_aggregate(train_df, group_col, feat_cols, n_splits=5, seed=42):
    """使用KFold进行OOF聚合以生成组级文本特征。"""
    oof_dict = {}
    global_means = {}
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)

    for f in feat_cols:
        # 确保每一列只包含数值
        train_df[f] = train_df[f].apply(
            lambda x: x if isinstance(x, (float, int)) else float('nan')  # 非数值数据转为NaN
        )

        # 计算组级特征的均值
        oof = pd.Series(index=train_df.index, dtype=float)
        global_means[f] = train_df[f].mean()  # 全局均值

        for tr, val in kf.split(train_df):
            # 分组计算均值，处理 NaN
            grp_mean = train_df.iloc[tr].groupby(group_col)[f].mean()
            oof.iloc[val] = train_df.iloc[val][group_col].map(grp_mean).fillna(global_means[f])
        
        oof_dict[f] = oof.fillna(global_means[f])  # 若该组无数据，则使用全局均值

    return oof_dict

# ---------- 主流程 ----------
group_col = None
for cand in ['小区', '小区名', '板块', '板块名']:
    if cand in df.columns:
        group_col = cand
        break

if group_col is None:
    raise KeyError("未找到用于聚合的列（如 '小区' 或 '板块'），请确认列名。")

if 'Price' in df.columns:
    try:
        df['Price'] = df['Price'].astype(str).str.replace(',', '').str.replace('元', '').astype(float)
    except Exception:
        df['Price'] = pd.to_numeric(df['Price'].astype(str).str.replace(r'[^\d\.]', '', regex=True), errors='coerce')

# 新的文本列，用于分析
text_columns = ['核心卖点', '户型介绍', '周边配套', '交通出行']

# 为每一列创建情感分析结果的列
for col in text_columns:
    if col not in df.columns:
        raise KeyError(f"找不到列 {col}，请确认。")
    
    # 对每一列的文本进行处理和情感分析
    docs = df[col].fillna("").astype(str).map(normalize_text).tolist()  # 处理文本
    sent_res = [doc_sentiment_snownlp(t) for t in docs]  # 情感分析

    # 将结果分别存入 DataFrame
    df[f'sent_score_{col}'] = [r[0] for r in sent_res]  # 情感得分
    df[f'n_sent_pos_{col}'] = [r[1] for r in sent_res]   # 正面句数
    df[f'n_sent_neg_{col}'] = [r[2] for r in sent_res]   # 负面句数
    df[f'n_sents_{col}'] = [r[3] for r in sent_res]      # 总句数

# TF-IDF + SVD特征生成
print("正在生成 TF-IDF -> SVD 特征。")
all_docs = df[text_columns].fillna("").astype(str).apply(lambda x: ' '.join(x), axis=1).tolist()  # 合并所有列为一列
Xred, _, _ = tfidf_svd_fit_transform(all_docs)  # 生成特征
for i in range(Xred.shape[1]):
    df[f'tfidf_svd_{i}'] = Xred[:, i]  # 添加SVD特征

# ---------- OOF 聚合 ----------
doc_feats = ['sent_score_核心卖点', 'n_sent_pos_核心卖点', 'n_sent_neg_核心卖点', 'n_sents_核心卖点',
             'sent_score_户型介绍', 'n_sent_pos_户型介绍', 'n_sent_neg_户型介绍', 'n_sents_户型介绍',
             'sent_score_周边配套', 'n_sent_pos_周边配套', 'n_sent_neg_周边配套', 'n_sents_周边配套',
             'sent_score_交通出行', 'n_sent_pos_交通出行', 'n_sent_neg_交通出行', 'n_sents_交通出行']

if 'Price' in df.columns and df['Price'].notna().sum() > 0:
    print("数据包含 Price，使用 KFold OOF 生成组级文本特征（CV-safe）...")
    oof_dict = oof_group_aggregate(df, group_col, doc_feats)
    for f, s in oof_dict.items():
        df[f'group_{f}'] = s.values
else:
    print("无 Price，直接用全表 group mean 映射（非 OOF），注意风险。")
    for f in doc_feats:
        mapping = df.groupby(group_col)[f].mean().to_dict()
        df[f'group_{f}'] = df[group_col].map(mapping).fillna(df[f].mean())

# 最终保存结果
output_file = "text_features_with_sentiment.csv"  # 输出文件名
df.to_csv(output_file, index=False)  # 将结果保存为 CSV
print(f"文本特征及情感分析结果已生成并保存为 {output_file}.")

In [ ]:
import pandas as pd

# 读取CSV文件
file_path = 'text_features_with_sentiment.csv'
data = pd.read_csv(file_path)

# 需要处理的列名
columns_to_encode = [
    "建筑结构_comm",
    "产权描述",
    "供水",
    "供暖",
    "供电",
    "物业类别",   
]

# 标签编码
for column in columns_to_encode:
    if column in data.columns:
        data[column], _ = pd.factorize(data[column])

# 保存到新的CSV文件
data.to_csv('c.csv', index=False, encoding='utf_8_sig')

print("数据量化完成并已保存至 'c.csv'")

In [ ]:
import pandas as pd

# 加载 CSV 文件
df = pd.read_csv('c.csv')  # 替换为你的文件名

# 要删除的列名列表
columns_to_drop = [
    '核心卖点',
    '户型介绍',
    '周边配套',
    '交通出行',
    '物业类别',
    '开发商',
    '物业公司',
    '客户反馈',
    '物业办公电话',
    'Unnamed: 0',
    '停车费用',
    '交易时间',
    '上次交易'
]

# 删除指定列
df.drop(columns=columns_to_drop, inplace=True, errors='ignore')  # 使用 errors='ignore' 忽略不存在的列

# 验证删除结果
print("删除后的列名:", df.columns)

# 可选：将修改后的 DataFrame 保存到新文件
df.to_csv('d.csv', index=False)  # 保存为新的 CSV 文件

print("已保存至 'd.csv'")

In [ ]:
import pandas as pd
import numpy as np

# 加载数据
file_path = "d.csv"  # 新的输入文件名
data = pd.read_csv(file_path)

# 显示数据的基本信息
print(data.info())

# 假设建筑年代的列名是 "建筑年代"
# 1. 计算均值或中位数
# 这里我们选择使用中位数进行填补
median_year = data['建筑年代'].median()

# 2. 用中位数填补缺失值
data['建筑年代'].fillna(median_year, inplace=True)

# 3. 检查是否填补成功
print("填补后的建筑年代列:")
print(data['建筑年代'].describe())

# 保存清洗后的数据到新的CSV文件
cleaned_file_path = "e.csv"  # 你可以根据需要修改输出文件名
data.to_csv(cleaned_file_path, index=False)
print(f"清洗后的数据已保存到 {cleaned_file_path}")

In [ ]:
import os
import re
import unicodedata
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import KFold

# 尝试导入 SnowNLP 库用于情感分析
try:
    from snownlp import SnowNLP
    _HAVE_SNOW = True
except ImportError:
    _HAVE_SNOW = False

# ---------- 加载数据 ----------
fname = "ruc_Class25Q2_test_price.csv"  # 数据文件名
if not os.path.exists(fname):
    raise FileNotFoundError(f"{fname} 不存在，请确认文件路径。")

# 尝试读取 CSV 文件，优先使用 UTF-8 编码，如果失败则使用 GBK 编码
try:
    df = pd.read_csv(fname, dtype=str, low_memory=False)
except Exception:
    df = pd.read_csv(fname, dtype=str, encoding='gbk', low_memory=False)

# ---------- 基础函数 ----------
def normalize_text(x):
    """对文本进行规范化处理，包括去除多余的空白字符和特殊字符。"""
    if pd.isna(x): 
        return ""
    s = str(x)
    s = unicodedata.normalize("NFKC", s)  # 将字符规范化
    s = re.sub(r'[\u200B-\u200F\uFEFF]', '', s)  # 去除不可见字符
    return re.sub(r'\s+', ' ', s).strip()  # 简化空格并去掉首尾空格

def doc_sentiment_snownlp(text):
    """计算文本的情感分数及句子统计信息。"""
    if not _HAVE_SNOW:
        return 0.5, 0, 0, 0  # 若未安装 SnowNLP，返回默认值
    s = SnowNLP(text)
    sentiment_score = s.sentiments  # 获取情感得分
    return sentiment_score, int(sentiment_score > 0.6), int(sentiment_score < 0.4), 1  # 返回情感分数和句子统计

# ---------- 特征提取 ----------
def tfidf_svd_fit_transform(texts, max_features=5000, n_components=20, random_state=42):
    """生成TF-IDF特征并进行SVD降维。"""
    tf = TfidfVectorizer(max_features=max_features, token_pattern=r"(?u)\b\w+\b")
    Xtf = tf.fit_transform(texts)  # 进行TF-IDF转换
    svd = TruncatedSVD(n_components=n_components, random_state=random_state)  # SVD降维
    Xred = svd.fit_transform(Xtf)  # 进行SVD变换
    return Xred, tf, svd  # 返回降维后的特征和模型

def oof_group_aggregate(train_df, group_col, feat_cols, n_splits=5, seed=42):
    """使用KFold进行OOF聚合以生成组级文本特征。"""
    oof_dict = {}
    global_means = {}
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)

    for f in feat_cols:
        # 确保每一列只包含数值
        train_df[f] = train_df[f].apply(
            lambda x: x if isinstance(x, (float, int)) else float('nan')  # 非数值数据转为NaN
        )

        # 计算组级特征的均值
        oof = pd.Series(index=train_df.index, dtype=float)
        global_means[f] = train_df[f].mean()  # 全局均值

        for tr, val in kf.split(train_df):
            # 分组计算均值，处理 NaN
            grp_mean = train_df.iloc[tr].groupby(group_col)[f].mean()
            oof.iloc[val] = train_df.iloc[val][group_col].map(grp_mean).fillna(global_means[f])
        
        oof_dict[f] = oof.fillna(global_means[f])  # 若该组无数据，则使用全局均值

    return oof_dict

# ---------- 主流程 ----------
group_col = None
for cand in ['小区', '小区名', '板块', '板块名']:
    if cand in df.columns:
        group_col = cand
        break

if group_col is None:
    raise KeyError("未找到用于聚合的列（如 '小区' 或 '板块'），请确认列名。")

if 'Price' in df.columns:
    try:
        df['Price'] = df['Price'].astype(str).str.replace(',', '').str.replace('元', '').astype(float)
    except Exception:
        df['Price'] = pd.to_numeric(df['Price'].astype(str).str.replace(r'[^\d\.]', '', regex=True), errors='coerce')

txt_col = '客户反馈'
if txt_col not in df.columns:
    raise KeyError(f"找不到列 {txt_col}，请确认。")

# 生成基础文本特征
docs = df[txt_col].fillna("").astype(str).map(normalize_text).tolist()

# 情感分析与句子统计
sent_res = [doc_sentiment_snownlp(t) for t in docs]
df['txt_sent_score'] = [r[0] for r in sent_res]  # 情感得分
df['txt_n_sent_pos'] = [r[1] for r in sent_res]   # 正面句数
df['txt_n_sent_neg'] = [r[2] for r in sent_res]   # 负面句数
df['txt_n_sents'] = [r[3] for r in sent_res]      # 总句数

# 文本长度特征
df['txt_len_chars'] = df[txt_col].fillna("").astype(str).str.len()  # 字符数
df['txt_len_words'] = df[txt_col].fillna("").astype(str).str.count(r'\w+')  # 词数（简单近似）

# TF-IDF + SVD特征生成
print("正在生成 TF-IDF -> SVD 特征。")
Xred, _, _ = tfidf_svd_fit_transform(docs)  # 生成特征
for i in range(Xred.shape[1]):
    df[f'tfidf_svd_{i}'] = Xred[:, i]  # 添加SVD特征

# ---------- OOF 聚合 ----------
doc_feats = ['txt_sent_score', 'txt_n_sent_pos', 'txt_n_sent_neg', 'txt_n_sents', 'txt_len_chars', 'txt_len_words']

if 'Price' in df.columns and df['Price'].notna().sum() > 0:
    print("数据包含 Price，使用 KFold OOF 生成组级文本特征（CV-safe）...")
    oof_dict = oof_group_aggregate(df, group_col, doc_feats)
    for f, s in oof_dict.items():
        df[f'group_{f}'] = s.values
else:
    print("无 Price，直接用全表 group mean 映射（非 OOF），注意风险。")
    for f in doc_feats:
        mapping = df.groupby(group_col)[f].mean().to_dict()
        df[f'group_{f}'] = df[group_col].map(mapping).fillna(df[f].mean())

# 最终保存结果
output_file = "ruc_text_features_with_sentiment.csv"  # 输出文件名
df.to_csv(output_file, index=False)  # 将结果保存为 CSV
print(f"文本特征及情感分析结果已生成并保存为 {output_file}.")


In [4]:
import pandas as pd
import numpy as np

# 加载数据
file_path = "ruc_text_features_with_sentiment.csv"  # 新的输入文件名
data = pd.read_csv(file_path)

# 显示数据的前几行和数据基本信息
print(data.head())
print(data.info())

# 1. 处理缺失值
# 检查缺失值
missing_values = data.isnull().sum()
print("缺失值统计：\n", missing_values)

# 对于数值型数据：用均值填补
for column in data.select_dtypes(include=[np.number]).columns:
    data[column].fillna(data[column].mean(), inplace=True)

# 对于文本型数据：用'未知'填补
for column in data.select_dtypes(include=[object]).columns:
    data[column].fillna('未知', inplace=True)

# 对于日期型数据：用最近的有效日期填补
for column in data.select_dtypes(include=[np.datetime64]).columns:
    data[column].fillna(data[column].max(), inplace=True)

# 2. 删除空白和无效值
data.replace("", np.nan, inplace=True)

# 3. 处理乱码
# 假设如有特定列可能存在编码问题，进行清理：
# data['column_name'] = data['column_name'].str.encode('utf-8').str.decode('utf-8')

# 4. 标准化数据格式
# 标准化日期格式
for column in data.select_dtypes(include=[np.datetime64]).columns:
    data[column] = pd.to_datetime(data[column], errors='coerce')


# 6. 删除重复数据
data.drop_duplicates(inplace=True)

# 7. 数据类型转换
# 确保数值、日期和文本数据的类型正确
for column in data.select_dtypes(include=[object]).columns:
    data[column] = data[column].astype(str)

# 8. 清洗完成后，重新检查数据
print("清洗后的数据:")
print(data.info())
print(data.head())

# 保存清洗后的数据到新的CSV文件
cleaned_file_path = "cleaned_data.csv"  # 你可以根据需要修改输出文件名
data.to_csv(cleaned_file_path, index=False)
print(f"清洗后的数据已保存到 {cleaned_file_path}")


C:\Users\86130\AppData\Local\Temp\ipykernel_31348\334368695.py:6: DtypeWarning: Columns (4,32) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(file_path)


        ID  城市     区域      板块    环线      房屋户型        所在楼层     建筑面积     套内面积  \
0  1000000   0  109.0   367.0  二至三环  3室2厅1厨2卫  中楼层 (共23层)  282.02㎡      NaN   
1  1000001   0   28.0   606.0  五至六环  2室1厅1厨1卫  中楼层 (共17层)   88.42㎡   71.78㎡   
2  1000002   0  123.0  1110.0  五至六环  3室1厅1厨2卫  高楼层 (共12层)  175.52㎡  139.86㎡   
3  1000003   0   65.0   555.0   六环外  2室1厅1厨1卫   中楼层 (共5层)  106.13㎡      NaN   
4  1000004   0  109.0   990.0   二环内  3室2厅1厨2卫    顶层 (共5层)   116.8㎡      NaN   

  房屋朝向  ... tfidf_svd_16 tfidf_svd_17 tfidf_svd_18 tfidf_svd_19  \
0  南 北  ...    -0.011374     0.021266     0.001508     0.009869   
1  南 北  ...     0.008697     0.000567    -0.005646    -0.001297   
2   西北  ...    -0.039646     0.045264    -0.138974    -0.076021   
3  南 北  ...     0.122109    -0.054036    -0.027674    -0.032792   
4  南 北  ...    -0.049761    -0.036195     0.016146    -0.014714   

  group_txt_sent_score group_txt_n_sent_pos group_txt_n_sent_neg  \
0             0.503457             0.407407           

C:\Users\86130\AppData\Local\Temp\ipykernel_31348\334368695.py:19: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data[column].fillna(data[column].mean(), inplace=True)
C:\Users\86130\AppData\Local\Temp\ipykernel_31348\334368695.py:23: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

F

清洗后的数据:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34017 entries, 0 to 34016
Data columns (total 87 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   ID                    34017 non-null  int64  
 1   城市                    34017 non-null  int64  
 2   区域                    34017 non-null  float64
 3   板块                    34017 non-null  float64
 4   环线                    34017 non-null  object 
 5   房屋户型                  34017 non-null  object 
 6   所在楼层                  34017 non-null  object 
 7   建筑面积                  34017 non-null  object 
 8   套内面积                  34017 non-null  object 
 9   房屋朝向                  34017 non-null  object 
 10  建筑结构                  34017 non-null  object 
 11  装修情况                  34017 non-null  object 
 12  梯户比例                  34017 non-null  object 
 13  配备电梯                  34017 non-null  object 
 14  别墅类型                  34017 non-null  object 
 15  交易时间       

In [5]:
import pandas as pd

# 读取CSV文件
file_path = 'cleaned_data.csv'
data = pd.read_csv(file_path)

# 需要处理的列名
columns_to_encode = [
    "环线",
    "房屋户型",
    "所在楼层",
    "房屋朝向",
    "建筑结构",
    "装修情况",
    "梯户比例",
    "配备电梯",
    "别墅类型",
    "交易权属",
    "房屋用途",
    "房屋年限",
    "抵押信息",
    "房屋优势",
    "环线位置",
    "产权所属",
]

# 标签编码
for column in columns_to_encode:
    if column in data.columns:
        data[column], _ = pd.factorize(data[column])

# 保存到新的CSV文件
data.to_csv('encod_cleaned_data.csv', index=False, encoding='utf_8_sig')

print("数据量化完成并已保存至 'encod_cleaned_data.csv'")

数据量化完成并已保存至 'encod_cleaned_data.csv'


In [6]:
import pandas as pd

# 读取CSV文件
file_path = 'encod_cleaned_data.csv'
data = pd.read_csv(file_path)

# 处理“套内面积”列
def process_area(area):
    if area == '未知' or pd.isna(area) or area == '':
        return 0  # 将“未知”和空字符串替换为0
    else:
        # 移除“m²”单位并转为浮点数
        try:
            return float(area[:-2])  # 去掉最后两个字符
        except ValueError:
            return 0  # 如果转换失败，返回0

# 应用处理函数
data['套内面积'] = data['套内面积'].apply(process_area)

# 打印结果以确认
print(data['套内面积'].head())

# 保存处理后的数据到新的CSV文件
data.to_csv('processed_cleaned_data.csv', index=False, encoding='utf_8_sig')

print("套内面积处理完成并已保存至 'processed_cleaned_data.csv'")

0      0.0
1     71.7
2    139.8
3      0.0
4      0.0
Name: 套内面积, dtype: float64
套内面积处理完成并已保存至 'processed_cleaned_data.csv'


In [7]:
import pandas as pd

# 读取CSV文件
file_path = 'processed_cleaned_data.csv'
data = pd.read_csv(file_path)

# 时间处理
def process_date(date_str):
    try:
        return pd.to_datetime(date_str)
    except (ValueError, TypeError):
        return pd.NaT

data['交易时间'] = data['交易时间'].apply(process_date)  # 将‘交易时间’转换为datetime
data['上次交易'] = data['上次交易'].apply(process_date)  # 将‘上次交易’转换为datetime

# 处理上次交易的缺失值
# 假设缺失的上次交易视为当前交易时间的上一个时间
data['上次交易'].fillna(data['交易时间'], inplace=True)

# 创建一个新列，标记上次交易是否缺失
data['上次交易缺失'] = data['上次交易'].isna().astype(int)

# 特征提取
data['交易年'] = data['交易时间'].dt.year
data['交易月'] = data['交易时间'].dt.month
data['交易日'] = data['交易时间'].dt.day
data['交易星期'] = data['交易时间'].dt.weekday

# 计算上次交易距离
data['上次交易距离'] = (data['交易时间'] - data['上次交易']).dt.days

# 打印结果以确认
print(data[['交易时间', '上次交易', '上次交易缺失', '上次交易距离']].head())

# 保存处理后的数据
data.to_csv('processed_with_time_features.csv', index=False, encoding='utf_8_sig')

print("时间特征处理完成并已保存至 'processed_with_time_features.csv'")


C:\Users\86130\AppData\Local\Temp\ipykernel_31348\2086269572.py:19: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['上次交易'].fillna(data['交易时间'], inplace=True)


        交易时间       上次交易  上次交易缺失  上次交易距离
0 2025-02-21 2008-02-02       0    6229
1 2025-01-17 2010-12-18       0    5144
2 2025-04-03 2010-05-07       0    5445
3 2025-01-31 2008-07-15       0    6044
4 2025-01-22 1999-10-26       0    9220
时间特征处理完成并已保存至 'processed_with_time_features.csv'


In [8]:
import pandas as pd

# 读取CSV文件
file_path = 'processed_with_time_features.csv'
data = pd.read_csv(file_path)

def process_area(area):
        try:
            return float(area[:-2])  # 去掉最后两个字符
        except ValueError:
            return 0  # 如果转换失败，返回0

# 应用处理函数
data['建筑面积'] = data['建筑面积'].apply(process_area)

# 打印结果以确认
print(data['建筑面积'].head())

# 保存处理后的数据到新的CSV文件
data.to_csv('new.csv', index=False, encoding='utf_8_sig')

print("处理完成并已保存至 'new.csv'")

0    282.0
1     88.4
2    175.5
3    106.1
4    116.0
Name: 建筑面积, dtype: float64
处理完成并已保存至 'new.csv'


In [9]:
import pandas as pd

# 读取CSV文件
file_path = 'new.csv'
data = pd.read_csv(file_path)

def process_area(area):
    if area == '未知':
        return 0  # 将“未知”替换为0
    else:
        # 移除“m²”单位并转为浮点数
        try:
            return float(area[:-1])  # 去掉最后字符
        except ValueError:
            return 0  # 如果转换失败，返回0

# 应用处理函数
data['房屋总数'] = data['房屋总数'].apply(process_area)
data['楼栋总数'] = data['楼栋总数'].apply(process_area)
data['绿 化 率'] = data['绿 化 率'].apply(process_area)


# 保存处理后的数据到新的CSV文件
data.to_csv('n.csv', index=False, encoding='utf_8_sig')

print("处理完成并已保存至 'n.csv'")

处理完成并已保存至 'n.csv'


In [10]:
import pandas as pd
import re

# 读取CSV文件
file_path = 'n.csv'  # 假设正确的文件路径
data = pd.read_csv(file_path)

# 处理建筑年代
def process_architecture_years(year_str):
    # 使用正则表达式提取年份
    years = re.findall(r'\d{4}', year_str)
    
    if len(years) == 1:  # 仅有一个年份
        return int(years[0])  # 返回单一年份
    elif len(years) == 2:  # 有两个年份
        return int((int(years[0]) + int(years[1])) / 2)  # 返回中间值
    else:
        return None  # 如果没有年份，返回None

# 应用处理函数
data['建筑年代'] = data['建筑年代'].apply(process_architecture_years)

# 打印结果以确认
print(data[['建筑年代']].head())

# 保存处理后的数据
data.to_csv('a.csv')
print("建筑年代处理完成并已保存至 'a.csv'")


     建筑年代
0  2004.0
1  2008.0
2  2000.0
3  2006.0
4  1995.0
建筑年代处理完成并已保存至 'a.csv'


In [11]:
import pandas as pd
import re

# 读取CSV文件
file_path = 'a.csv'  # 假设正确的文件路径
data = pd.read_csv(file_path)

# 处理物业费
def process_property_fee(fee_str):
    # 检查是否为“未知”
    if fee_str.strip().lower() == "未知":
        return 0  # 将“未知”视为0
    
    # 提取所有的数值 (整数或小数)
    numbers = re.findall(r'\d*\.?\d+', fee_str)  # 提取所有数字
    
    # 转换为浮点数
    numbers = [float(num) for num in numbers]
    
    if len(numbers) == 1:  # 仅有一个数值
        return numbers[0]  # 返回该数值
    elif len(numbers) == 2:  # 有两个数值
        return sum(numbers) / 2  # 返回中间值
    else:
        return 0  # 如果无法匹配，有其他情况，返回0

# 应用处理函数
data['物 业 费'] = data['物 业 费'].apply(process_property_fee)
data['燃气费'] = data['燃气费'].apply(process_property_fee)
data['供热费'] = data['供热费'].apply(process_property_fee)

# 保存处理后的数据
data.to_csv('b.csv', index=False, encoding='utf_8_sig')

print("物业费处理完成并已保存至 'b.csv'")


物业费处理完成并已保存至 'b.csv'


In [12]:
import os
import re
import unicodedata
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import KFold

# 尝试导入 SnowNLP 库用于情感分析
try:
    from snownlp import SnowNLP
    _HAVE_SNOW = True
except ImportError:
    _HAVE_SNOW = False

# ---------- 加载数据 ----------
fname = "b.csv"  # 数据文件名
if not os.path.exists(fname):
    raise FileNotFoundError(f"{fname} 不存在，请确认文件路径。")

# 尝试读取 CSV 文件，优先使用 UTF-8 编码，如果失败则使用 GBK 编码
try:
    df = pd.read_csv(fname, dtype=str, low_memory=False)
except Exception:
    df = pd.read_csv(fname, dtype=str, encoding='gbk', low_memory=False)

# ---------- 基础函数 ----------
def normalize_text(x):
    """对文本进行规范化处理，包括去除多余的空白字符和特殊字符。"""
    if pd.isna(x): 
        return ""
    s = str(x)
    s = unicodedata.normalize("NFKC", s)  # 将字符规范化
    s = re.sub(r'[\u200B-\u200F\uFEFF]', '', s)  # 去除不可见字符
    return re.sub(r'\s+', ' ', s).strip()  # 简化空格并去掉首尾空格

def doc_sentiment_snownlp(text):
    """计算文本的情感分数及句子统计信息。"""
    if not _HAVE_SNOW:
        return 0.5, 0, 0, 0  # 若未安装 SnowNLP，返回默认值
    s = SnowNLP(text)
    sentiment_score = s.sentiments  # 获取情感得分
    return sentiment_score, int(sentiment_score > 0.6), int(sentiment_score < 0.4), 1  # 返回情感分数和句子统计

# ---------- 特征提取 ----------
def tfidf_svd_fit_transform(texts, max_features=5000, n_components=20, random_state=42):
    """生成TF-IDF特征并进行SVD降维。"""
    tf = TfidfVectorizer(max_features=max_features, token_pattern=r"(?u)\b\w+\b")
    Xtf = tf.fit_transform(texts)  # 进行TF-IDF转换
    svd = TruncatedSVD(n_components=n_components, random_state=random_state)  # SVD降维
    Xred = svd.fit_transform(Xtf)  # 进行SVD变换
    return Xred, tf, svd  # 返回降维后的特征和模型

def oof_group_aggregate(train_df, group_col, feat_cols, n_splits=5, seed=42):
    """使用KFold进行OOF聚合以生成组级文本特征。"""
    oof_dict = {}
    global_means = {}
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)

    for f in feat_cols:
        # 确保每一列只包含数值
        train_df[f] = train_df[f].apply(
            lambda x: x if isinstance(x, (float, int)) else float('nan')  # 非数值数据转为NaN
        )

        # 计算组级特征的均值
        oof = pd.Series(index=train_df.index, dtype=float)
        global_means[f] = train_df[f].mean()  # 全局均值

        for tr, val in kf.split(train_df):
            # 分组计算均值，处理 NaN
            grp_mean = train_df.iloc[tr].groupby(group_col)[f].mean()
            oof.iloc[val] = train_df.iloc[val][group_col].map(grp_mean).fillna(global_means[f])
        
        oof_dict[f] = oof.fillna(global_means[f])  # 若该组无数据，则使用全局均值

    return oof_dict

# ---------- 主流程 ----------
group_col = None
for cand in ['小区', '小区名', '板块', '板块名']:
    if cand in df.columns:
        group_col = cand
        break

if group_col is None:
    raise KeyError("未找到用于聚合的列（如 '小区' 或 '板块'），请确认列名。")

if 'Price' in df.columns:
    try:
        df['Price'] = df['Price'].astype(str).str.replace(',', '').str.replace('元', '').astype(float)
    except Exception:
        df['Price'] = pd.to_numeric(df['Price'].astype(str).str.replace(r'[^\d\.]', '', regex=True), errors='coerce')

# 新的文本列，用于分析
text_columns = ['核心卖点', '户型介绍', '周边配套', '交通出行']

# 为每一列创建情感分析结果的列
for col in text_columns:
    if col not in df.columns:
        raise KeyError(f"找不到列 {col}，请确认。")
    
    # 对每一列的文本进行处理和情感分析
    docs = df[col].fillna("").astype(str).map(normalize_text).tolist()  # 处理文本
    sent_res = [doc_sentiment_snownlp(t) for t in docs]  # 情感分析

    # 将结果分别存入 DataFrame
    df[f'sent_score_{col}'] = [r[0] for r in sent_res]  # 情感得分
    df[f'n_sent_pos_{col}'] = [r[1] for r in sent_res]   # 正面句数
    df[f'n_sent_neg_{col}'] = [r[2] for r in sent_res]   # 负面句数
    df[f'n_sents_{col}'] = [r[3] for r in sent_res]      # 总句数

# TF-IDF + SVD特征生成
print("正在生成 TF-IDF -> SVD 特征。")
all_docs = df[text_columns].fillna("").astype(str).apply(lambda x: ' '.join(x), axis=1).tolist()  # 合并所有列为一列
Xred, _, _ = tfidf_svd_fit_transform(all_docs)  # 生成特征
for i in range(Xred.shape[1]):
    df[f'tfidf_svd_{i}'] = Xred[:, i]  # 添加SVD特征

# ---------- OOF 聚合 ----------
doc_feats = ['sent_score_核心卖点', 'n_sent_pos_核心卖点', 'n_sent_neg_核心卖点', 'n_sents_核心卖点',
             'sent_score_户型介绍', 'n_sent_pos_户型介绍', 'n_sent_neg_户型介绍', 'n_sents_户型介绍',
             'sent_score_周边配套', 'n_sent_pos_周边配套', 'n_sent_neg_周边配套', 'n_sents_周边配套',
             'sent_score_交通出行', 'n_sent_pos_交通出行', 'n_sent_neg_交通出行', 'n_sents_交通出行']

if 'Price' in df.columns and df['Price'].notna().sum() > 0:
    print("数据包含 Price，使用 KFold OOF 生成组级文本特征（CV-safe）...")
    oof_dict = oof_group_aggregate(df, group_col, doc_feats)
    for f, s in oof_dict.items():
        df[f'group_{f}'] = s.values
else:
    print("无 Price，直接用全表 group mean 映射（非 OOF），注意风险。")
    for f in doc_feats:
        mapping = df.groupby(group_col)[f].mean().to_dict()
        df[f'group_{f}'] = df[group_col].map(mapping).fillna(df[f].mean())

# 最终保存结果
output_file = "text_features_with_sentiment.csv"  # 输出文件名
df.to_csv(output_file, index=False)  # 将结果保存为 CSV
print(f"文本特征及情感分析结果已生成并保存为 {output_file}.")

正在生成 TF-IDF -> SVD 特征。
无 Price，直接用全表 group mean 映射（非 OOF），注意风险。
文本特征及情感分析结果已生成并保存为 text_features_with_sentiment.csv.


In [13]:
import pandas as pd

# 读取CSV文件
file_path = 'text_features_with_sentiment.csv'
data = pd.read_csv(file_path)

# 需要处理的列名
columns_to_encode = [
    "建筑结构_comm",
    "产权描述",
    "供水",
    "供暖",
    "供电",
    "物业类别",   
]

# 标签编码
for column in columns_to_encode:
    if column in data.columns:
        data[column], _ = pd.factorize(data[column])

# 保存到新的CSV文件
data.to_csv('c.csv', index=False, encoding='utf_8_sig')

print("数据量化完成并已保存至 'c.csv'")

数据量化完成并已保存至 'c.csv'


In [15]:
import pandas as pd

# 加载 CSV 文件
df = pd.read_csv('c.csv')  # 替换为你的文件名

# 要删除的列名列表
columns_to_drop = [
    '核心卖点',
    '户型介绍',
    '周边配套',
    '交通出行',
    '物业类别',
    '开发商',
    '物业公司',
    '客户反馈',
    '物业办公电话',
    'Unnamed: 0',
    '停车费用',
    '交易时间',
    '上次交易',
    'ID'
    
]

# 删除指定列
df.drop(columns=columns_to_drop, inplace=True, errors='ignore')  # 使用 errors='ignore' 忽略不存在的列

# 验证删除结果
print("删除后的列名:", df.columns)

# 可选：将修改后的 DataFrame 保存到新文件
df.to_csv('d1.csv', index=False)  # 保存为新的 CSV 文件

print("已保存至 'd1.csv'")

删除后的列名: Index(['ID', '城市', '区域', '板块', '环线', '房屋户型', '所在楼层', '建筑面积', '套内面积', '房屋朝向',
       ...
       'group_n_sent_neg_户型介绍', 'group_n_sents_户型介绍', 'group_sent_score_周边配套',
       'group_n_sent_pos_周边配套', 'group_n_sent_neg_周边配套', 'group_n_sents_周边配套',
       'group_sent_score_交通出行', 'group_n_sent_pos_交通出行',
       'group_n_sent_neg_交通出行', 'group_n_sents_交通出行'],
      dtype='object', length=113)
已保存至 'd1.csv'


In [1]:
import pandas as pd
import numpy as np

# 加载数据
file_path = "d1.csv"  # 新的输入文件名
data = pd.read_csv(file_path)

# 显示数据的基本信息
print(data.info())

# 假设建筑年代的列名是 "建筑年代"
# 1. 计算均值或中位数
# 这里我们选择使用中位数进行填补
median_year = data['建筑年代'].median()

# 2. 用中位数填补缺失值
data['建筑年代'].fillna(median_year, inplace=True)

# 3. 检查是否填补成功
print("填补后的建筑年代列:")
print(data['建筑年代'].describe())

# 保存清洗后的数据到新的CSV文件
cleaned_file_path = "e1.csv"  # 你可以根据需要修改输出文件名
data.to_csv(cleaned_file_path, index=False)
print(f"清洗后的数据已保存到 {cleaned_file_path}")


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34017 entries, 0 to 34016
Columns: 113 entries, ID to group_n_sents_交通出行
dtypes: float64(67), int64(46)
memory usage: 29.3 MB
None
填补后的建筑年代列:
count    34017.000000
mean      2005.413058
std          8.432109
min       1936.000000
25%       2003.000000
50%       2007.000000
75%       2010.000000
max       2022.000000
Name: 建筑年代, dtype: float64


C:\Users\86130\AppData\Local\Temp\ipykernel_6604\3454213882.py:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['建筑年代'].fillna(median_year, inplace=True)


清洗后的数据已保存到 e1.csv


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# 1. 数据加载
print("Loading data...")
data = pd.read_csv('e.csv')

# 2. 数据处理
print("Processing data...")
if 'community_price' in data.columns:
    data.drop(columns=['community_price'], inplace=True)

# a. 处理缺失值
imputer = SimpleImputer(strategy='mean')
data[:] = imputer.fit_transform(data)

# b. 检测并处理异常值
for column in data.select_dtypes(include=[np.number]).columns:
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    data = data[(data[column] >= lower_bound) & (data[column] <= upper_bound)]

# 3. 特征和目标分离
X = data.drop(columns=['Price'])
y = data['Price'].clip(lower=0)

# 4. 划分训练集和测试集
print("Splitting data into train and test sets...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=111)

# 5. 标准化特征
print("Scaling features...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 6. 建模
print("Training models...")
models = {
    'OLS': LinearRegression(),
    'Lasso': Lasso(alpha=0.01),
    'Ridge': Ridge(alpha=1.0),
    'ElasticNet': ElasticNet(alpha=0.01, l1_ratio=0.5)
}

results = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)

    # 预测
    y_pred = model.predict(X_test_scaled)
    
    # 修正负值
    y_pred[y_pred < 0] = 0
    y_pred[np.isinf(y_pred)] = 0  # 修正无穷值

    # 计算样本外 MAE
    sample_out_mae = mean_absolute_error(y_test, y_pred)
    results[name] = {
        'Sample Out MAE': sample_out_mae,
        'Train MAE': mean_absolute_error(y_train, model.predict(X_train_scaled)),
        'CV MAE': -cross_val_score(model, X_train_scaled, y_train, cv=6, scoring='neg_mean_absolute_error').mean()
    }

# 7. 汇总结果
output_data = {
    'Metrics': [],
    'In-sample': [],
    'Out-of-sample': [],
    'Cross-validation': [],
    'Kaggle Score': []
}

# 使用示例的Kaggle得分
kaggle_scores = [60, 61, 62, 62]  # 根据具体情况调整

for i, name in enumerate(results.keys()):
    output_data['Metrics'].append(name)
    output_data['In-sample'].append(results[name]['Train MAE'])
    output_data['Out-of-sample'].append(results[name]['Sample Out MAE'])
    output_data['Cross-validation'].append(results[name]['CV MAE'])
    output_data['Kaggle Score'].append(kaggle_scores[i])

# 创建 DataFrame
output_df = pd.DataFrame(output_data)

# 打印结果
print(output_df)

# 8. 保存预测结果
results_df = pd.DataFrame({
    'Actual': y_test,
    'OLS_Predicted': np.clip(models['OLS'].predict(X_test_scaled), 0, None),
    'Lasso_Predicted': np.clip(models['Lasso'].predict(X_test_scaled), 0, None),
    'Ridge_Predicted': np.clip(models['Ridge'].predict(X_test_scaled), 0, None),
    'ElasticNet_Predicted': np.clip(models['ElasticNet'].predict(X_test_scaled), 0, None),
})
results_df.to_csv('results.csv', index=False)
print("预测结果已保存到 'results.csv'")
import joblib
joblib.dump(models['OLS'], 'OLS_model.joblib')
joblib.dump(models['Lasso'], 'Lasso_model.joblib')
joblib.dump(models['Ridge'], 'Ridge_model.joblib')
joblib.dump(models['ElasticNet'], 'ElasticNet_model.joblib')


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import joblib

# 1. 加载新数据
print("Loading new data...")
new_data = pd.read_csv('e1.csv')


# 3. 加载之前训练好的模型
print("Loading trained models...")
models = {
   
    'ElasticNet': joblib.load('ElasticNet_model.joblib')
}

# 4. 特征标准化
scaler = StandardScaler()
new_data_scaled = scaler.fit_transform(new_data)

# 5. 进行预测并修正负值
predictions = {}
for name, model in models.items():
    y_pred = model.predict(new_data_scaled)
    y_pred[y_pred < 0] = 0  # 修正负值
    predictions[name] = y_pred

# 6. 创建结果数据框
results_df = pd.DataFrame(predictions)

# 7. 保存预测结果
results_df.to_csv('价格预测.csv', index=False)
print("预测结果已保存")
